# MusicAI – ACE-Step 1.5 nhẹ trên Google Colab

Notebook gồm đúng **2 ô mã**: chạy **Ô 1** xong mới chạy **Ô 2**.

- Nếu Colab cấp GPU, notebook tự dùng CUDA.
- Nếu hết hạn mức GPU, notebook tự chạy bằng CPU (chậm hơn nhiều).
- Vẫn tạo được nhạc có lời từ lời nhập sẵn, nhạc không lời, bản 30 giây và 180 giây.
- 5Hz LM được tắt để giảm RAM/VRAM; chức năng AI gợi ý lời sẽ do MusicAI xử lý riêng.
- Model được giữ trong Google Drive để phiên sau không phải tải lại toàn bộ.

> Muốn dùng GPU: chọn `Thời gian chạy → Thay đổi loại thời gian chạy → T4 GPU`. Nếu Colab từ chối GPU, chọn CPU và chạy tiếp.

In [ ]:
# Ô 1 — CÀI ĐẶT / KHÔI PHỤC MÔI TRƯỜNG
from google.colab import drive
from pathlib import Path
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')

repo = Path('/content/ACE-Step-1.5')
drive_root = Path('/content/drive/MyDrive/MusicAI_ACE_Step')
drive_checkpoints = drive_root / 'checkpoints'
uv_cache = Path('/content/uv-cache')

drive_checkpoints.mkdir(parents=True, exist_ok=True)
uv_cache.mkdir(parents=True, exist_ok=True)
# Không đặt cache uv trên Drive: filesystem Drive dễ gây lỗi lock/symlink khi uv sync.
os.environ['UV_CACHE_DIR'] = str(uv_cache)
os.environ['ACESTEP_CHECKPOINTS_DIR'] = str(drive_checkpoints)

if not (repo / '.git').exists():
    if repo.exists():
        shutil.rmtree(repo)
    print('Đang tải mã nguồn ACE-Step 1.5...')
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/ace-step/ACE-Step-1.5.git',
        str(repo)
    ], check=True)
else:
    print('Mã nguồn đã có trong runtime, bỏ qua git clone.')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
os.chdir(repo)
def run_visible(command):
    print('>', ' '.join(map(str, command)), flush=True)
    result = subprocess.run(command)
    if result.returncode != 0:
        raise RuntimeError(
            f"Lệnh thất bại (mã {result.returncode}): {' '.join(map(str, command))}"
        )

# Nếu lần chạy trước để lại .venv dở, xóa rồi cài sạch một lần.
venv = repo / '.venv'
sync_command = ['uv', 'sync', '--python', '3.12']
first_sync = subprocess.run(sync_command)
if first_sync.returncode != 0:
    print('uv sync lần đầu thất bại. Đang xóa .venv dở và thử lại một lần...')
    if venv.exists():
        shutil.rmtree(venv)
    run_visible(sync_command)

# Chuyển thư mục model sang Google Drive.
local_checkpoints = repo / 'checkpoints'
if local_checkpoints.is_symlink():
    local_checkpoints.unlink()
elif local_checkpoints.exists():
    for child in list(local_checkpoints.iterdir()):
        target = drive_checkpoints / child.name
        if not target.exists():
            shutil.move(str(child), str(target))
        elif child.is_dir():
            shutil.rmtree(child)
        else:
            child.unlink()
    shutil.rmtree(local_checkpoints)
local_checkpoints.symlink_to(drive_checkpoints, target_is_directory=True)

# Tự bỏ qua model đã có trên Drive.
run_visible(['uv', 'run', '--no-sync', 'acestep-download'])

nvidia_smi = shutil.which('nvidia-smi')
gpu = None
gpu_name = ''
if nvidia_smi:
    gpu = subprocess.run(
        [nvidia_smi, '--query-gpu=name,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True
    )
    if gpu.returncode == 0:
        gpu_name = gpu.stdout.strip()

# Tesla T4 từng sinh NaN với float16 trong một số phiên Colab.
# Chỉ vá T4; GPU khác giữ cấu hình tự động để không tăng VRAM không cần thiết.
if 'T4' in gpu_name.upper():
    patch_file = repo / 'acestep/core/generation/handler/init_service_orchestrator.py'
    if patch_file.exists():
        source = patch_file.read_text(encoding='utf-8')
        if 'self.dtype = torch.float16' in source:
            source = source.replace(
                'self.dtype = torch.float16',
                'self.dtype = torch.float32',
                1
            )
            patch_file.write_text(source, encoding='utf-8')
            print('Đã bật float32 ổn định cho Tesla T4.')

if gpu is not None and gpu.returncode == 0:
    print('SẴN SÀNG — GPU:', gpu_name)
else:
    print('SẴN SÀNG — không có GPU, Ô 2 sẽ chạy bằng CPU.')
print('Tiếp tục chạy Ô 2.')

In [ ]:
# Ô 2 — KHỞI ĐỘNG MÁY CHỦ (giữ ô này chạy trong suốt lúc kiểm thử)
import os
import subprocess
import shutil

os.chdir('/content/ACE-Step-1.5')
nvidia_smi = shutil.which('nvidia-smi')
gpu = subprocess.run([nvidia_smi], capture_output=True) if nvidia_smi else None
device = 'cuda' if gpu is not None and gpu.returncode == 0 else 'cpu'
gpu_name = ''
if device == 'cuda':
    info = subprocess.run(
        [nvidia_smi, '--query-gpu=name', '--format=csv,noheader'],
        capture_output=True, text=True
    )
    gpu_name = info.stdout.strip()

os.environ['MPLBACKEND'] = 'Agg'
os.environ['ACESTEP_CHECKPOINTS_DIR'] = '/content/drive/MyDrive/MusicAI_ACE_Step/checkpoints'
os.environ['ACESTEP_DEVICE'] = device
os.environ['ACESTEP_LM_DEVICE'] = device
os.environ['ACESTEP_CONFIG_PATH'] = 'acestep-v15-turbo'
os.environ['ACESTEP_INIT_LLM'] = 'false'
os.environ['ACESTEP_LM_BACKEND'] = 'pt'
os.environ['ACESTEP_API_WORKERS'] = '1'
os.environ['ACESTEP_QUEUE_WORKERS'] = '1'
os.environ['ACESTEP_USE_FLASH_ATTENTION'] = 'false'
os.environ['ACESTEP_DTYPE'] = 'float32' if device == 'cpu' or 'T4' in gpu_name.upper() else 'float16'

if device == 'cuda':
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    os.environ['ACESTEP_OFFLOAD_TO_CPU'] = 'true'
    offload = 'true'
    print('CHẾ ĐỘ GPU: CUDA + CPU offload, batch 1, không dùng 5Hz LM.')
else:
    os.environ['ACESTEP_OFFLOAD_TO_CPU'] = 'false'
    offload = 'false'
    print('CHẾ ĐỘ CPU: bài 180 giây có thể mất rất nhiều thời gian.')

command = [
    'uv', 'run', 'acestep',
    '--share',
    '--backend', 'pt',
    '--init_service', 'true',
    '--init_llm', 'false',
    '--config_path', 'acestep-v15-turbo',
    '--batch_size', '1',
    '--offload_to_cpu', offload,
    '--enable-api'
]

print('Đang khởi động ACE-Step...')
print('Chờ dòng Running on public URL rồi mở liên kết gradio.live.')
result = subprocess.run(command[:2] + ['--no-sync'] + command[2:])
if result.returncode != 0:
    raise RuntimeError(f'ACE-Step đã dừng với mã lỗi {result.returncode}.')

## Cấu hình kiểm thử sau khi mở Gradio

### Bản 30 giây có lời

- `Generation Mode`: Custom
- Bỏ chọn `Instrumental`, `Thinking`, `AutoGen`
- `Audio Duration`: 30
- `Batch Size`: 1
- `Inference Steps`: 4; chất lượng chưa tốt thì tăng lên 8
- `Vocal Language`: vi hoặc auto

Music Caption giọng nữ:

```text
Vietnamese acoustic pop, clear young female vocal, vocals begin immediately, short intro, warm guitar, gentle piano, catchy chorus
```

Lyrics:

```text
[Verse]
Ngày mới lên qua ô cửa nhỏ
Mang theo hy vọng trong tim

[Chorus]
Mình cùng đi qua bao giông gió
Giữ mãi thanh âm bình yên
```

Muốn giọng nam, đổi `female vocal` thành `male vocal`.

### Bản đầy đủ có lời

- `Audio Duration`: 180
- `Batch Size`: 1
- `Inference Steps`: 8
- Lời cần đủ Verse, Chorus, Verse 2, Bridge, Final Chorus và Outro.

### Nhạc không lời

- Chọn `Instrumental`
- Lyrics: `[Instrumental]`
- Thời lượng: 30 hoặc 180 giây

> CPU fallback vẫn tạo nhạc có lời nhưng rất chậm. Luôn kiểm thử 30 giây trước và không gửi hai yêu cầu cùng lúc.